Step 1: Load and visualize optical CSV data

In [2]:
%matplotlib qt
import csv
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

EXPERIMENTS_DIR = Path(r'C:\Users\javie\Desktop\Universidad\Verano 2026\laser_correction')
DEFAULT_EXPERIMENT_NAME = 'Experiment 4 070126'
DEFAULT_REGION_NAME = 'Region 1'

# â”€â”€ User input â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
experiment_name = input(f"Enter experiment folder name [{DEFAULT_EXPERIMENT_NAME}]: ").strip() or DEFAULT_EXPERIMENT_NAME
region_name     = input(f"Enter region folder name [{DEFAULT_REGION_NAME}]: ").strip() or DEFAULT_REGION_NAME
experiment_dir  = EXPERIMENTS_DIR / experiment_name / region_name
assert experiment_dir.is_dir(), f"Folder not found: {experiment_dir}"

optical_files = sorted(experiment_dir.glob('*Optical.csv'))
assert len(optical_files) > 0, f"No *Optical.csv found in {experiment_dir}"
assert len(optical_files) == 1, (
    "Multiple *Optical.csv found â€” rename to disambiguate:\n" +
    "\n".join(f"  {f.name}" for f in optical_files)
)

csv_path = optical_files[0]
print(f"Experiment  : {EXPERIMENTS_DIR / experiment_name}")
print(f"Region      : {region_name}")
print(f"Optical CSV : {csv_path.name}")

# â”€â”€ Read all rows â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def read_csv_rows(path: Path) -> list:
    """Read every row from the CSV, handling Windows CRLF line endings."""
    with open(path, "r", encoding="utf-8", errors="replace", newline="") as f:
        return list(csv.reader(f))


rows = read_csv_rows(csv_path)
print(f"Total rows in file: {len(rows)}")

# â”€â”€ Parse header metadata â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def parse_metadata(rows: list) -> dict:
    """
    Extract scalar metadata from the fixed header block.
    All row indices are 0-based (spreadsheet row = index + 1).

    Row 12  ->  XY Calibration: pixel size in Âµm
    Row 14  ->  Horizontal:     declared column count (includes 2 sentinel cols)
    Row 15  ->  Vertical:       image height in pixels
    """
    pixel_size_um    = float(rows[12][1])
    width_declared   = int(rows[14][1])   # includes 2 sentinel border columns
    height           = int(rows[15][1])
    return {
        "pixel_size_um":  pixel_size_um,
        "width_declared": width_declared,
        "height":         height,
    }


meta = parse_metadata(rows)
print(f"Pixel size (declared) : {meta['pixel_size_um']} Âµm")
print(f"Width (declared)      : {meta['width_declared']} cols  "
      f"â†’  {meta['width_declared'] - 2} image pixels")
print(f"Height                : {meta['height']} px")

# â”€â”€ Locate the three channel blocks â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def find_channel_start(rows: list, label: str) -> int:
    """
    Return the 0-based index of the first DATA row immediately after the
    channel label row (e.g. 'Monochrome (R)').
    """
    for i, row in enumerate(rows):
        if row and row[0].strip() == label:
            return i + 1
    raise ValueError(f"Label '{label}' not found in file.")


r_start = find_channel_start(rows, "Monochrome (R)")
g_start = find_channel_start(rows, "Monochrome (G)")
b_start = find_channel_start(rows, "Monochrome (B)")

h = meta["height"]

# Actual pixel width: each CSV row = 2 sentinel columns + image pixels
# Infer from the first data row to be robust across file variants.
actual_w = len(rows[r_start]) - 2

print(f"Actual pixel dimensions : {actual_w} w Ã— {h} h")
print(f"R data rows (0-based)   : {r_start} â€“ {r_start + h - 1}")
print(f"G data rows             : {g_start} â€“ {g_start + h - 1}")
print(f"B data rows             : {b_start} â€“ {b_start + h - 1}")

# â”€â”€ Extract a single channel â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def extract_channel(rows: list, start: int, height: int, width: int) -> np.ndarray:
    """
    Parse `height` CSV rows starting at row index `start` into a uint8 array
    of shape (height, width).

    Column layout per row:  [ sentinel_0 | px_0 â€¦ px_{w-1} | sentinel_last ]
    Image pixels occupy columns 1 through width (inclusive, 0-based).

    Returns
    -------
    np.ndarray  shape (height, width), dtype uint8
    """
    channel = np.empty((height, width), dtype=np.uint8)
    for i in range(height):
        channel[i] = np.array(rows[start + i][1 : width + 1], dtype=np.uint8)
    return channel


print("Extracting channels â€¦", end="", flush=True)
R = extract_channel(rows, r_start, h, actual_w)
print(" R", end="", flush=True)
G = extract_channel(rows, g_start, h, actual_w)
print(" G", end="", flush=True)
B = extract_channel(rows, b_start, h, actual_w)
print(" B â€” done.")

# Stack into a single (height, width, 3) RGB image array
rgb_image = np.stack([R, G, B], axis=-1)
print(f"RGB array : shape={rgb_image.shape}, dtype={rgb_image.dtype}")
print(f"Value ranges â€” R: {R.min()}â€“{R.max()}  "
      f"G: {G.min()}â€“{G.max()}  B: {B.min()}â€“{B.max()}")

# â”€â”€ Display the image â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
pixel_size = meta["pixel_size_um"]

# Physical extent for axis tick labels [x_left, x_right, y_bottom, y_top] in mm
extent_mm = [
    0,
    actual_w * pixel_size / 1000,
    h       * pixel_size / 1000,
    0,
]

fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(rgb_image, aspect="equal", extent=extent_mm, origin="upper")
ax.set_xlabel("x (mm)")
ax.set_ylabel("y (mm)")
ax.set_title(
    f"Optical scan â€” {csv_path.name}\n"
    f"{actual_w} Ã— {h} px  |  {pixel_size} Âµm/px"
)
plt.tight_layout()
plt.show()

phys_w_mm = actual_w * pixel_size / 1000
phys_h_mm = h        * pixel_size / 1000
print(f"Physical dimensions : {phys_w_mm:.2f} mm Ã— {phys_h_mm:.2f} mm")

Experiment  : C:\Users\javie\Desktop\Universidad\Verano 2026\laser_correction\Experiment 4 070126
Region      : Region 1
Optical CSV : Region 1 Pass 1_Optical.csv
Total rows in file: 4635
Pixel size (declared) : 11.814 Âµm
Width (declared)      : 2049 cols  â†’  2047 image pixels
Height                : 1537 px
Actual pixel dimensions : 2047 w Ã— 1537 h
R data rows (0-based)   : 22 â€“ 1558
G data rows             : 1560 â€“ 3096
B data rows             : 3098 â€“ 4634
Extracting channels â€¦ R G B â€” done.
RGB array : shape=(1537, 2047, 3), dtype=uint8
Value ranges â€” R: 0â€“255  G: 0â€“255  B: 0â€“255
Physical dimensions : 24.18 mm Ã— 18.16 mm


Step 2: Load and visualize height data â€” adjust the colormap slider to reveal surface topography

In [3]:
%matplotlib qt
from matplotlib.widgets import Slider

# â”€â”€ Auto-locate height CSV in the same experiment folder â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
height_files = sorted(experiment_dir.glob('*Height.csv'))
assert len(height_files) > 0, f"No *Height.csv found in {experiment_dir}"
assert len(height_files) == 1, (
    "Multiple *Height.csv found â€” rename to disambiguate:\n" +
    "\n".join(f"  {f.name}" for f in height_files)
)

height_csv_path = height_files[0]
print(f"Height CSV  : {height_csv_path.name}")

# â”€â”€ Parse height metadata and verify it matches the optical scan â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
height_rows = read_csv_rows(height_csv_path)   # reuse function from Step 1
height_meta = parse_metadata(height_rows)       # same header layout

print(f"Height pixel size : {height_meta['pixel_size_um']} Âµm")
print(f"Height grid size  : {height_meta['width_declared']} Ã— {height_meta['height']}")

assert height_meta['pixel_size_um'] == meta['pixel_size_um'], \
    "Pixel size mismatch between optical and height files!"
assert height_meta['height'] == meta['height'], \
    "Row count mismatch between optical and height files!"

print("\nâœ“ Pixel size and grid dimensions match the optical scan.")

# â”€â”€ Extract height array â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def extract_height(
    rows: list,
    label: str = "Height",
    width: int = None,
) -> tuple:
    """
    Extract the height channel from a profilometer height CSV.

    Column layout differs from the optical format:
        optical : [ 0_sentinel | px0 â€¦ px_{w-1} | 0_sentinel ]  (2+w cols)
        height  : [ px0 â€¦ px_{w-1} | empty_sentinel ]            (1+w cols)

    Missing / unmeasured pixels are encoded as empty strings and returned
    as NaN in the output array.

    Parameters
    ----------
    rows  : all CSV rows (from read_csv_rows)
    label : channel label to locate (default 'Height')
    width : expected pixel width; if None, inferred from the first data row

    Returns
    -------
    Z           : np.ndarray, shape (height, width), dtype float32, NaN where unmeasured
    unit        : str, height unit read from the file header (e.g. 'mm')
    z_min, z_max: float, declared min/max from the header
    """
    # Locate the data block
    data_start = find_channel_start(rows, label)   # reuse from Step 1
    h_px = int(rows[15][1])                         # Vertical

    # Height rows: cols 0..w-1 are pixels; the very last column is always empty.
    # Infer pixel width from the first data row (total cols minus 1 trailing empty).
    if width is None:
        width = len(rows[data_start]) - 1

    # Scalar metadata unique to the height file
    unit  = rows[18][1]         # row 19  â†’  'mm'
    z_min = float(rows[16][1])  # row 17
    z_max = float(rows[17][1])  # row 18

    # Build the array; empty strings â†’ NaN
    Z = np.full((h_px, width), np.nan, dtype=np.float32)
    for i in range(h_px):
        row = rows[data_start + i]
        for j in range(width):
            v = row[j]
            if v != '':
                Z[i, j] = float(v)

    return Z, unit, z_min, z_max


print("Extracting height data â€¦", end=" ", flush=True)
Z, z_unit, z_min_hdr, z_max_hdr = extract_height(height_rows, width=actual_w)
print("done.")

nan_frac = np.mean(np.isnan(Z)) * 100
print(f"Array shape   : {Z.shape}")
print(f"NaN (missing) : {nan_frac:.2f}%")
print(f"Value range   : {np.nanmin(Z):.4f} to {np.nanmax(Z):.4f} {z_unit}")
print(f"Header range  : {z_min_hdr:.4f} to {z_max_hdr:.4f} {z_unit}")

# â”€â”€ Colormap statistics â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
z_median = np.nanmedian(Z)
z_std    = np.nanstd(Z)
n_sigma_init = 0.15
print(f"Median : {z_median:.4f} {z_unit}  |  Std : {z_std:.4f} {z_unit}")

# â”€â”€ Visualise: optical image + height map side by side â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
extent_mm = [
    0,
    actual_w * pixel_size / 1000,
    h        * pixel_size / 1000,
    0,
]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
plt.subplots_adjust(bottom=0.12)   # room for slider

# Left: optical
axes[0].imshow(rgb_image, aspect="equal", extent=extent_mm, origin="upper")
axes[0].set_title("Optical scan")
axes[0].set_xlabel("x (mm)")
axes[0].set_ylabel("y (mm)")

# Right: height map â€” midpoint = median, range = n_sigma
im = axes[1].imshow(
    Z,
    aspect="equal",
    extent=extent_mm,
    origin="upper",
    cmap="viridis",
    vmin=z_median - n_sigma_init * z_std,
    vmax=z_median + n_sigma_init * z_std,
)
axes[1].set_title(f"Height map ({z_unit})")
axes[1].set_xlabel("x (mm)")
axes[1].set_ylabel("y (mm)")
plt.colorbar(im, ax=axes[1], label=f"Height ({z_unit})", shrink=0.85)

fig.suptitle(csv_path.stem, fontsize=13)

# â”€â”€ Slider: colormap range in units of Ïƒ (applied symmetrically around median)
ax_slider = fig.add_axes([0.2, 0.03, 0.6, 0.03])
slider = Slider(
    ax=ax_slider,
    label="Colormap range (Â± NÂ·Ïƒ)",
    valmin=0.05,
    valmax=5,
    valinit=n_sigma_init,
    valstep=0.01,
)

def _update_clim(val):
    n = slider.val
    im.set_clim(z_median - n * z_std, z_median + n * z_std)
    fig.canvas.draw_idle()

slider.on_changed(_update_clim)

plt.show()

Height CSV  : Region 1 Pass 1_Height.csv
Height pixel size : 11.814 Âµm
Height grid size  : 2049 Ã— 1537

âœ“ Pixel size and grid dimensions match the optical scan.
Extracting height data â€¦ done.
Array shape   : (1537, 2047)
NaN (missing) : 1.38%
Value range   : -0.1410 to 0.1320 mm
Header range  : -0.1410 to 0.1320 mm
Median : 0.0000 mm  |  Std : 0.0201 mm


Step 3: Overlay the height data onto the optical data, adjust the slider to change visibility

In [4]:
%matplotlib qt
import matplotlib.colors as mcolors
from matplotlib.widgets import Slider

# â”€â”€ Inherit exact colormap range from Block 2's Â± NÂ·Ïƒ slider â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# im.norm reflects whatever clim the user last set via the slider above.
vmin_overlay = im.norm.vmin
vmax_overlay = im.norm.vmax
print(f"Height colormap range from Block 2 slider: "
      f"{vmin_overlay:.4f} to {vmax_overlay:.4f} {z_unit}  "
      f"(â‰ˆ Â±{(vmax_overlay - vmin_overlay) / (2 * z_std):.2f} Ïƒ)")

# â”€â”€ Normalize optical image to float [0, 1] â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
optical_float = rgb_image.astype(np.float32) / 255.0      # (H, W, 3)

# â”€â”€ Map height data to RGBA using the same colormap and range â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
cmap_height = plt.get_cmap("viridis")
norm_overlay = mcolors.Normalize(vmin=vmin_overlay, vmax=vmax_overlay, clip=True)

height_rgba = cmap_height(norm_overlay(np.where(np.isnan(Z), 0.0, Z))).astype(np.float32)
height_rgba[np.isnan(Z), 3] = 0.0   # NaN â†’ fully transparent; always shows optical

# â”€â”€ Composite function â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def make_composite(alpha: float) -> np.ndarray:
    """alpha=0 â†’ pure optical, alpha=1 â†’ pure height colormap."""
    h_alpha = height_rgba[..., 3:4] * alpha
    blended = h_alpha * height_rgba[..., :3] + (1.0 - h_alpha) * optical_float
    return np.clip(blended, 0.0, 1.0)

# â”€â”€ Figure â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
alpha_init = 0.5
fig, ax = plt.subplots(figsize=(11, 8))
plt.subplots_adjust(bottom=0.15)

im_overlay = ax.imshow(
    make_composite(alpha_init),
    aspect="equal",
    extent=extent_mm,
    origin="upper",
    interpolation="nearest",
)

sm = plt.cm.ScalarMappable(cmap=cmap_height, norm=norm_overlay)
sm.set_array([])
fig.colorbar(sm, ax=ax, label=f"Height ({z_unit})", shrink=0.85)

ax.set_xlabel("x (mm)")
ax.set_ylabel("y (mm)")
ax.set_title(f"Optical + Height overlay â€” {csv_path.stem}")

# â”€â”€ Blend slider â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
ax_slider = fig.add_axes([0.18, 0.05, 0.64, 0.03])
slider_blend = Slider(
    ax=ax_slider,
    label="Optical  â†  blend  â†’  Height",
    valmin=0.0,
    valmax=1.0,
    valinit=alpha_init,
    valstep=0.01,
)

def _on_blend_change(val):
    im_overlay.set_data(make_composite(slider_blend.val))
    fig.canvas.draw_idle()

slider_blend.on_changed(_on_blend_change)
plt.show()

Height colormap range from Block 2 slider: -0.0030 to 0.0030 mm  (â‰ˆ Â±0.15 Ïƒ)


Step 4: Match a line to a known mark â€” a perpendicular line of equal length auto-appears on the left; use arrow keys to translate and Shift+â†/â†’ to rotate the pair

In [7]:
%matplotlib qt
import sys
import numpy as np
from matplotlib.lines import Line2D
from contextlib import nullcontext

# stdout context for Qt callbacks; no extra widget package needed.
_out6 = nullcontext()

# â”€â”€ Render overlay â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fig, ax = plt.subplots(figsize=(11, 8))
plt.subplots_adjust(top=0.88)

ax.imshow(
    make_composite(slider_blend.val),
    aspect="equal",
    extent=extent_mm,
    origin="upper",
    interpolation="nearest",
)
ax.set_xlabel("x (mm)")
ax.set_ylabel("y (mm)")

# â”€â”€ Line artists â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
rubber_l1 = Line2D([], [], color='white', lw=1.5, ls='--', alpha=0.8)
rubber_l2 = Line2D([], [], color='cyan',  lw=1.5, ls='--', alpha=0.8)
line1_art = Line2D([], [], color='yellow', lw=2.0, alpha=0.95)
line2_art = Line2D([], [], color='cyan',   lw=2.0, alpha=0.95)

pt1_dot, = ax.plot([], [], 'wo', ms=5, zorder=5)
pt2_dot, = ax.plot([], [], 'yo', ms=5, zorder=5)

for artist in (rubber_l1, rubber_l2, line1_art, line2_art):
    ax.add_line(artist)

title = ax.set_title("Click to place line 1 â€” start point")

# â”€â”€ Exported results (available to following cells) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
line1_coords = {'p1': None, 'p2': None}
line2_coords = {'p1': None, 'p2': None}

# â”€â”€ State â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
#   phase 0 : awaiting line 1 start click
#   phase 1 : awaiting line 1 end click       (rubber_l1 tracks cursor)
#   phase 2 : awaiting line 2 lower-end click (rubber_l2 anchored at line 1 p1)
#   phase 3 : translate / rotate / enter
state = {'phase': 0, 'p1': None, 'cids': []}

TRANSLATE_STEP_MM = pixel_size / 1000
ROTATE_STEP_DEG   = 0.1

# â”€â”€ Helpers â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def _redraw():
    p1, p2 = line1_coords['p1'], line1_coords['p2']
    q1, q2 = line2_coords['p1'], line2_coords['p2']
    line1_art.set_data([p1[0], p2[0]], [p1[1], p2[1]])
    line2_art.set_data([q1[0], q2[0]], [q1[1], q2[1]])
    fig.canvas.draw_idle()

def _update_title():
    p1, p2 = line1_coords['p1'], line1_coords['p2']
    q1, q2 = line2_coords['p1'], line2_coords['p2']
    title.set_text(
        f"Arrow keys: translate  |  Shift+â†/â†’: rotate  |  Enter: confirm\n"
        f"L1({p1[0]:.3f}, {p1[1]:.3f})â†’({p2[0]:.3f}, {p2[1]:.3f})  "
        f"L2({q1[0]:.3f}, {q1[1]:.3f})â†’({q2[0]:.3f}, {q2[1]:.3f}) mm"
    )

def _translate(dx, dy):
    for coords in (line1_coords, line2_coords):
        coords['p1'] = (coords['p1'][0] + dx, coords['p1'][1] + dy)
        coords['p2'] = (coords['p2'][0] + dx, coords['p2'][1] + dy)
    _redraw()
    _update_title()

def _rotate(angle_deg):
    angle = np.deg2rad(angle_deg)
    c, s = np.cos(angle), np.sin(angle)
    R = np.array([[c, -s], [s, c]])
    pivot = (np.array(line1_coords['p1']) + np.array(line1_coords['p2'])) / 2
    for coords in (line1_coords, line2_coords):
        for key in ('p1', 'p2'):
            v = np.array(coords[key]) - pivot
            coords[key] = tuple(pivot + R @ v)
    _redraw()
    _update_title()

# â”€â”€ Event handlers â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def on_motion(event):
    if event.inaxes is not ax:
        return
    if state['phase'] == 1:
        rubber_l1.set_data([state['p1'][0], event.xdata], [state['p1'][1], event.ydata])
        fig.canvas.draw_idle()
    elif state['phase'] == 2:
        q1 = line1_coords['p1']
        rubber_l2.set_data([q1[0], event.xdata], [q1[1], event.ydata])
        fig.canvas.draw_idle()

def on_click(event):
    if event.inaxes is not ax or event.button != 1:
        return
    x, y = event.xdata, event.ydata

    if state['phase'] == 0:
        state['p1'] = (x, y)
        state['phase'] = 1
        pt1_dot.set_data([x], [y])
        title.set_text("Click to place line 1 â€” end point")
        fig.canvas.draw_idle()

    elif state['phase'] == 1:
        p1, p2 = state['p1'], (x, y)
        line1_coords['p1'], line1_coords['p2'] = p1, p2
        rubber_l1.set_data([], [])
        pt2_dot.set_data([x], [y])
        line1_art.set_data([p1[0], p2[0]], [p1[1], p2[1]])
        state['phase'] = 2
        title.set_text(
            f"Click to place line 2 â€” lower endpoint  "
            f"(upper end fixed at line 1 start: {p1[0]:.3f}, {p1[1]:.3f} mm)"
        )
        fig.canvas.draw_idle()

    elif state['phase'] == 2:
        q1 = line1_coords['p1']   # upper end: fixed to line 1's p1
        q2 = (x, y)
        line2_coords['p1'] = q1
        line2_coords['p2'] = q2
        rubber_l2.set_data([], [])
        state['phase'] = 3
        _redraw()
        _update_title()

def on_key(event):
    if state['phase'] != 3:
        return
    step = TRANSLATE_STEP_MM
    if event.key == '8':
        _translate(0, -step)
    elif event.key == '2':
        _translate(0,  step)
    elif event.key == '4':
        _translate(-step, 0)
    elif event.key == '6':
        _translate( step, 0)
    elif event.key == '9':
        _rotate( ROTATE_STEP_DEG)
    elif event.key == '7':
        _rotate(-ROTATE_STEP_DEG)
    elif event.key == 'enter':
        p1, p2 = line1_coords['p1'], line1_coords['p2']
        q1, q2 = line2_coords['p1'], line2_coords['p2']
        l2_len_mm = np.linalg.norm(np.array(q2) - np.array(q1))
        title.set_text(
            f"Confirmed\n"
            f"L1({p1[0]:.3f},{p1[1]:.3f})â†’({p2[0]:.3f},{p2[1]:.3f})  "
            f"L2({q1[0]:.3f},{q1[1]:.3f})â†’({q2[0]:.3f},{q2[1]:.3f}) mm"
        )
        fig.canvas.draw_idle()
        for cid in state['cids']:
            fig.canvas.mpl_disconnect(cid)
        angle_deg = np.degrees(np.arctan2(p2[1] - p1[1], p2[0] - p1[0]))
        with _out6:
            print(f"Line 1          : {line1_coords['p1']} -> {line1_coords['p2']}  mm")
            print(f"Line 2          : {line2_coords['p1']} -> {line2_coords['p2']}  mm")
            print(f"Line 2 length   : {l2_len_mm:.4f} mm  ({l2_len_mm*1000:.2f} um)")
            print(f"Line 1 angle    : {angle_deg:.3f}Â° from +x axis  (positive = clockwise, y-down image coords)")

state['cids'] = [
    fig.canvas.mpl_connect('motion_notify_event', on_motion),
    fig.canvas.mpl_connect('button_press_event',  on_click),
    fig.canvas.mpl_connect('key_press_event',     on_key),
]

plt.show()

Step 5: Place a third line parallel to line 1 â€” drag it along the perpendicular axis and click to fix its position

In [8]:
%matplotlib qt
import numpy as np
from matplotlib.lines import Line2D
from contextlib import nullcontext

assert line1_coords['p1'] is not None, "Run Step 6 first."
assert line2_coords['p1'] is not None, "Run Step 6 first."

# stdout context for Qt callbacks; no extra widget package needed.
_out7 = nullcontext()

# â”€â”€ Re-open figure with Step 6 lines â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fig, ax = plt.subplots(figsize=(11, 8))
plt.subplots_adjust(top=0.88)

ax.imshow(
    make_composite(slider_blend.val),
    aspect="equal",
    extent=extent_mm,
    origin="upper",
    interpolation="nearest",
)
ax.set_xlabel("x (mm)")
ax.set_ylabel("y (mm)")

# â”€â”€ Redraw Step 6 lines â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
_l1p1 = np.array(line1_coords['p1'])
_l1p2 = np.array(line1_coords['p2'])
_l2q1 = np.array(line2_coords['p1'])
_l2q2 = np.array(line2_coords['p2'])

ax.add_line(Line2D([_l1p1[0], _l1p2[0]], [_l1p1[1], _l1p2[1]], color='yellow', lw=2.0, alpha=0.95))
ax.add_line(Line2D([_l2q1[0], _l2q2[0]], [_l2q1[1], _l2q2[1]], color='cyan',   lw=2.0, alpha=0.95))

# â”€â”€ Geometry â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
_d1 = _l1p2 - _l1p1
_u  = _d1 / np.linalg.norm(_d1)
_v  = _l2q2 - _l2q1
_L2 = np.linalg.norm(_v)

_det = _v[0]*_u[1] - _v[1]*_u[0]
_PARALLEL_THRESH = np.sin(np.deg2rad(5))

if abs(_det) / _L2 < _PARALLEL_THRESH:
    with _out7:
        print("Warning: Line 1 and Line 2 are nearly parallel (< 5Â°). "
              "Scan line placement will not work correctly.")

# â”€â”€ Intersection: line through cursor parallel to Line 1  Ã—  Line 2 â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def _intersect(cx, cy):
    if abs(_det) / _L2 < _PARALLEL_THRESH:
        return None
    dx = _l2q1[0] - cx
    dy = _l2q1[1] - cy
    s  = (_u[0]*dy - _u[1]*dx) / _det
    r_on_l2 = _l2q1 + s * _v
    r1 = tuple(r_on_l2)
    r2 = tuple(r_on_l2 + _d1)
    return r1, r2, float(s * _L2)

# â”€â”€ Rubber band and result storage â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
scan_rubber = Line2D([], [], color='lime', lw=2.0, alpha=0.95, ls='--')
ax.add_line(scan_rubber)

title = ax.set_title(
    "Move cursor to position scan line (constrained to line 2) â€” click to place"
)

scan_lines = []
state = {'sliding': True, 'cids': []}

# â”€â”€ Event handlers â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def on_motion(event):
    if event.inaxes is not ax or not state['sliding']:
        return
    result = _intersect(event.xdata, event.ydata)
    if result is None:
        return
    r1, r2, dist_mm = result
    scan_rubber.set_data([r1[0], r2[0]], [r1[1], r2[1]])
    title.set_text(
        f"Move cursor to position scan line â€” click to place  |  "
        f"Dist. along line 2: {dist_mm:+.4f} mm"
    )
    fig.canvas.draw_idle()

def on_click(event):
    if event.inaxes is not ax or event.button != 1 or not state['sliding']:
        return
    result = _intersect(event.xdata, event.ydata)
    if result is None:
        with _out7:
            print("Warning: lines nearly parallel â€” scan line not placed.")
        return
    r1, r2, dist_mm = result
    scan_rubber.set_data([], [])
    ax.add_line(Line2D([r1[0], r2[0]], [r1[1], r2[1]], color='lime', lw=2.0, alpha=0.95))
    fig.canvas.draw_idle()
    scan_lines.append({'p1': r1, 'p2': r2, 'distance': dist_mm})
    state['sliding'] = False
    n = len(scan_lines)
    title.set_text(
        f"Scan line {n} placed at {dist_mm:+.4f} mm along line 2  |  "
        f"N: place another  |  Enter: finish"
    )
    fig.canvas.draw_idle()
    with _out7:
        print(f"Scan line {n:2d} : ({r1[0]:.4f}, {r1[1]:.4f}) -> ({r2[0]:.4f}, {r2[1]:.4f}) mm")
        print(f"  Distance from Line 1 along Line 2: {dist_mm:.4f} mm")

def on_key(event):
    if event.key in ('n', 'N') and not state['sliding']:
        state['sliding'] = True
        title.set_text(
            f"Move cursor to position scan line {len(scan_lines) + 1} â€” click to place"
        )
        fig.canvas.draw_idle()
    elif event.key == 'enter' and not state['sliding']:
        title.set_text(f"Done â€” {len(scan_lines)} scan line(s) placed")
        fig.canvas.draw_idle()
        for cid in state['cids']:
            fig.canvas.mpl_disconnect(cid)
        with _out7:
            print(f"\nTotal scan lines placed: {len(scan_lines)}")

state['cids'] = [
    fig.canvas.mpl_connect('motion_notify_event', on_motion),
    fig.canvas.mpl_connect('button_press_event',  on_click),
    fig.canvas.mpl_connect('key_press_event',     on_key),
]

plt.show()

Step 6: Height profile along scan line â€” press + / - to widen the averaging region one pixel at a time

In [9]:
%matplotlib qt
import numpy as np
from scipy.ndimage import map_coordinates
from matplotlib.lines import Line2D
from matplotlib.widgets import Button, Slider

assert len(scan_lines) > 0, "Run Step 7 first to place at least one scan line."

px_per_mm  = 1000.0 / pixel_size
_Z_filled  = np.where(np.isnan(Z), 0.0, Z)
_Z_nanmask = np.isnan(Z).astype(np.float32)


def _sample_profile(sl, pad_px):
    p1m = np.array(sl['p1'], dtype=float)
    p2m = np.array(sl['p2'], dtype=float)
    p1p, p2p = p1m * px_per_mm, p2m * px_per_mm
    n   = max(int(np.ceil(np.linalg.norm(p2p - p1p))), 2)
    t   = np.linspace(0, 1, n)
    cb  = p1p[0] + t * (p2p[0] - p1p[0])
    rb  = p1p[1] + t * (p2p[1] - p1p[1])
    u   = (p2p - p1p) / np.linalg.norm(p2p - p1p)
    nc, nr = -u[1], u[0]
    rows = []
    for k in range(-pad_px, pad_px + 1):
        rk = np.clip(rb + k * nr, 0, Z.shape[0] - 1)
        ck = np.clip(cb + k * nc, 0, Z.shape[1] - 1)
        v  = map_coordinates(_Z_filled,  [rk, ck], order=1, mode='nearest')
        v[map_coordinates(_Z_nanmask, [rk, ck], order=0, mode='nearest') > 0.5] = np.nan
        rows.append(v)
    return t * float(np.linalg.norm(p2m - p1m)), np.nanmean(np.array(rows), axis=0)


for i, sl in enumerate(scan_lines):

    win = {'pad': 0, 'done': False}

    # â”€â”€ Layout â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    fig, (ax_img, ax_prof) = plt.subplots(
        1, 2, figsize=(16, 6),
        gridspec_kw={'width_ratios': [1, 2]},
    )
    plt.subplots_adjust(left=0.05, right=0.84, bottom=0.18, top=0.92, wspace=0.25)

    # Left: overlay thumbnail â€” active scan line bright, others dimmed
    ax_img.imshow(make_composite(slider_blend.val),
                  aspect='equal', extent=extent_mm, origin='upper',
                  interpolation='nearest')
    for cds, col in ((line1_coords, 'yellow'), (line2_coords, 'cyan')):
        ax_img.add_line(Line2D([cds['p1'][0], cds['p2'][0]],
                               [cds['p1'][1], cds['p2'][1]], color=col, lw=1.5))
    for j, s in enumerate(scan_lines):
        ax_img.add_line(Line2D([s['p1'][0], s['p2'][0]], [s['p1'][1], s['p2'][1]],
                               color='lime',
                               lw=2.5 if j == i else 1.0,
                               alpha=0.95 if j == i else 0.35))
    ax_img.set_xlabel("x (mm)", fontsize=8)
    ax_img.set_ylabel("y (mm)", fontsize=8)
    ax_img.set_title(f"Scan line {i + 1} of {len(scan_lines)}", fontsize=9)

    # Right: height profile
    d0, a0 = _sample_profile(sl, 0)
    (prof_line,) = ax_prof.plot(d0, a0, color='steelblue', lw=1.2)
    ax_prof.set_xlabel("Distance along scan line (mm)", fontsize=9)
    ax_prof.set_ylabel(f"Height ({z_unit})", fontsize=9)
    ax_prof.grid(True, alpha=0.3)

    def _title():
        n_px = 2 * win['pad'] + 1
        return (f"Scan line {i + 1} of {len(scan_lines)}  |  "
                f"Width: {n_px * pixel_size:.2f} Âµm ({n_px} px)")

    ax_prof.set_title(_title(), fontsize=9)

    # Pad slider (below profile panel)
    ax_sl  = fig.add_axes([0.42, 0.06, 0.38, 0.03])
    pad_sl = Slider(ax_sl, 'Avg width (px)', 0, 20, valinit=0, valstep=1,
                    color='steelblue')

    # Confirm button (right of profile panel)
    ax_btn = fig.add_axes([0.86, 0.44, 0.11, 0.09])
    btn    = Button(ax_btn, 'Confirm', color='#1e6e1e', hovercolor='#2ea82e')
    btn.label.set_color('white')
    btn.label.set_fontsize(10)

    # â”€â”€ Event handlers â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    def on_pad(val):
        win['pad'] = int(round(val))
        d, a = _sample_profile(sl, win['pad'])
        prof_line.set_data(d, a)
        ax_prof.relim()
        ax_prof.autoscale_view()
        ax_prof.set_title(_title(), fontsize=9)
        fig.canvas.draw_idle()

    def on_confirm(event):
        scan_lines[i]['pad'] = win['pad']
        win['done'] = True
        plt.close(fig)

    pad_sl.on_changed(on_pad)
    btn.on_clicked(on_confirm)

    plt.show()
    while plt.fignum_exists(fig.number):
        plt.pause(0.05)

    if win['done']:
        n_px = 2 * win['pad'] + 1
        print(f"Scan line {i + 1}: confirmed  pad={win['pad']} px "
              f"({n_px * pixel_size:.2f} Âµm averaging width)")
    else:
        print(f"Scan line {i + 1}: window closed without confirming.")

print(f"\nDone: {len(scan_lines)} scan line(s) processed.")

Scan line 1: confirmed  pad=0 px (11.81 Âµm averaging width)
Scan line 2: confirmed  pad=0 px (11.81 Âµm averaging width)
Scan line 3: confirmed  pad=0 px (11.81 Âµm averaging width)
Scan line 4: confirmed  pad=0 px (11.81 Âµm averaging width)
Scan line 5: confirmed  pad=0 px (11.81 Âµm averaging width)

Done: 5 scan line(s) processed.


Step 7: Slice the scan line profile into height levels and export DXF + SVG for SAMLight â€” drag thresholds, use + / - to add/remove levels, and assign Pen 1-14 per level

In [11]:
%matplotlib qt
import numpy as np
from scipy.ndimage import map_coordinates
from matplotlib.widgets import Button, TextBox, Slider
import ezdxf

assert len(scan_lines) > 0, "Run Steps 6 and 7 first."

px_per_mm  = 1000.0 / pixel_size
_Zf  = np.where(np.isnan(Z), 0.0, Z)
_Znm = np.isnan(Z).astype(np.float32)
CMAP = plt.cm.tab10

# SAMLight pen RGB table validated in explicacion_svg.md.
# Keep this embedded here so the notebook does not depend on the Qt tool or CSV exports.
SAMLIGHT_PEN_RGB = {
    1: (255, 0, 0),
    2: (0, 255, 0),
    3: (0, 0, 255),
    4: (0, 128, 255),
    5: (255, 170, 0),
    6: (0, 170, 170),
    7: (85, 85, 85),
    8: (255, 85, 0),
    9: (0, 170, 0),
    10: (255, 255, 0),
    11: (255, 0, 255),
    12: (0, 0, 85),
    13: (255, 170, 255),
    14: (170, 225, 255),
}
DXF_COLOR_CODES = [3, 6, 1, 4, 30, 5, 8, 30, 3, 4, 6, 5, 6, 4]
MIN_DXF_SEGMENT_MM = 0.055
SVG_STROKE_WIDTH_MM = 0.02
SVG_MARGIN_MM = 1.0


def _level_for_band(band_idx):
    # Notebook bands are exported as SAMLight levels N2, N3, ...
    # This matches the naming used by interfaz_calibracion_manual_qt.py.
    return int(band_idx) + 2


def _default_pen_for_level(level):
    return int(np.clip(int(level) - 1, 1, max(SAMLIGHT_PEN_RGB)))


def _clean_pen(value, default=1):
    try:
        pen = int(float(str(value).strip()))
    except (TypeError, ValueError):
        pen = int(default)
    return int(np.clip(pen, 1, max(SAMLIGHT_PEN_RGB)))


def _pen_rgb(pen):
    return SAMLIGHT_PEN_RGB[_clean_pen(pen)]


def _rgb_float(pen):
    r, g, b = _pen_rgb(pen)
    return (r / 255.0, g / 255.0, b / 255.0)


def _pen_dxf_color(pen):
    return int(DXF_COLOR_CODES[(_clean_pen(pen) - 1) % len(DXF_COLOR_CODES)])


def _svg_escape(value):
    return (str(value).replace('&', '&amp;')
                      .replace('"', '&quot;')
                      .replace('<', '&lt;')
                      .replace('>', '&gt;'))


def write_svg_lines(path, segments):
    if not segments:
        path.write_text(
            '<?xml version="1.0" encoding="UTF-8"?>\n'
            '<svg xmlns="http://www.w3.org/2000/svg" width="1mm" height="1mm" '
            'viewBox="0 0 1 1"></svg>\n',
            encoding='utf-8',
        )
        return

    xs = [float(v) for seg in segments for v in (seg['x0'], seg['x1'])]
    ys = [float(v) for seg in segments for v in (seg['y0'], seg['y1'])]
    x_min = min(xs) - SVG_MARGIN_MM
    x_max = max(xs) + SVG_MARGIN_MM
    y_min = min(ys) - SVG_MARGIN_MM
    y_max = max(ys) + SVG_MARGIN_MM
    width = max(x_max - x_min, 0.001)
    height = max(y_max - y_min, 0.001)

    lines = [
        '<?xml version="1.0" encoding="UTF-8"?>',
        '<svg xmlns="http://www.w3.org/2000/svg"',
        f'     width="{width:.6f}mm" height="{height:.6f}mm"',
        f'     viewBox="{x_min:.6f} {-y_max:.6f} {width:.6f} {height:.6f}">',
        f'  <g fill="none" stroke-width="{SVG_STROKE_WIDTH_MM:.6f}" stroke-linecap="round">',
    ]

    for idx, seg in enumerate(segments, start=1):
        r, g, b = _pen_rgb(seg['pen'])
        seg_id = seg.get('id') or f"{seg['layer']}_{idx:04d}"
        lines.append(
            f'    <line id="{_svg_escape(seg_id)}" '
            f'x1="{seg["x0"]:.6f}" y1="{-seg["y0"]:.6f}" '
            f'x2="{seg["x1"]:.6f}" y2="{-seg["y1"]:.6f}" '
            f'stroke="rgb({r},{g},{b})" />'
        )

    lines.extend(['  </g>', '</svg>'])
    path.write_text('\n'.join(lines) + '\n', encoding='utf-8')


def _sample_profile(sl, pad_px):
    p1m = np.array(sl['p1'], dtype=float)
    p2m = np.array(sl['p2'], dtype=float)
    p1p, p2p = p1m * px_per_mm, p2m * px_per_mm
    n   = max(int(np.ceil(np.linalg.norm(p2p - p1p))), 2)
    t   = np.linspace(0, 1, n)
    cb  = p1p[0] + t * (p2p[0] - p1p[0])
    rb  = p1p[1] + t * (p2p[1] - p1p[1])
    u   = (p2p - p1p) / np.linalg.norm(p2p - p1p)
    nc, nr = -u[1], u[0]
    rows = []
    for k in range(-pad_px, pad_px + 1):
        rk = np.clip(rb + k * nr, 0, Z.shape[0] - 1)
        ck = np.clip(cb + k * nc, 0, Z.shape[1] - 1)
        v  = map_coordinates(_Zf,  [rk, ck], order=1, mode='nearest')
        v[map_coordinates(_Znm, [rk, ck], order=0, mode='nearest') > 0.5] = np.nan
        rows.append(v)
    return t * float(np.linalg.norm(p2m - p1m)), np.nanmean(np.array(rows), axis=0)


def _classify(avg, thresholds):
    out = np.full(len(avg), -1, dtype=int)
    m   = ~np.isnan(avg)
    if m.any():
        out[m] = np.digitize(avg[m], sorted(thresholds))
    return out


def _build_segs(band_arr):
    segs, i = [], 0
    while i < len(band_arr):
        b = int(band_arr[i])
        j = i + 1
        while j < len(band_arr) and int(band_arr[j]) == b:
            j += 1
        segs.append((b, i, j))
        i = j
    return segs


def _build_segs_merged(band_arr, dist, min_len_mm):
    segs = []
    i = 0
    while i < len(band_arr):
        b = int(band_arr[i])
        j = i + 1
        while j < len(band_arr) and int(band_arr[j]) == b:
            j += 1
        segs.append([b, i, j])
        i = j

    changed = True
    while changed:
        changed = False
        for k in range(len(segs)):
            b, s, e = segs[k]
            if b < 0 or (dist[e - 1] - dist[s]) >= min_len_mm:
                continue
            has_prev = k > 0
            has_next = k < len(segs) - 1
            if has_prev and has_next:
                # Absorb into whichever neighbor has the lower band index
                if segs[k - 1][0] <= segs[k + 1][0]:
                    segs[k - 1][2] = e
                else:
                    segs[k + 1][1] = s
            elif has_prev:
                segs[k - 1][2] = e
            elif has_next:
                segs[k + 1][1] = s
            else:
                continue
            segs.pop(k)
            changed = True
            break
        merged = []
        for seg in segs:
            if merged and merged[-1][0] == seg[0]:
                merged[-1][2] = seg[2]
            else:
                merged.append(seg)
        segs = merged
    return segs


# â”€â”€ Part 1: per-line band-setting + pen-assignment loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
for i, sl in enumerate(scan_lines):

    pad_px     = sl.get('pad', 0)
    dist, avg  = _sample_profile(sl, pad_px)
    hv         = avg[~np.isnan(avg)]
    h_lo, h_hi = float(hv.min()), float(hv.max())
    h_rng      = h_hi - h_lo

    thresholds = list(sl.get('thresholds',
                             [h_lo + h_rng / 3, h_lo + 2 * h_rng / 3]))

    s      = {'min_len_mm': sl.get('min_len_mm', 0.05)}
    win    = {'done': False}
    drag   = {'idx': None}
    bstate = {'tbs': [], 'patch_axes': [], 'tb_axes': []}

    # â”€â”€ Layout â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    fig = plt.figure(figsize=(14, 9))

    ax_prof  = fig.add_axes([0.07, 0.47, 0.76, 0.46])   # top: profile
    ax_strip = fig.add_axes([0.07, 0.32, 0.76, 0.12])   # middle: band strip
    ax_sl_ax = fig.add_axes([0.07, 0.25, 0.76, 0.03])   # min-length slider
    ax_btn   = fig.add_axes([0.86, 0.47, 0.11, 0.46])   # confirm button
    # bottom region y=0.06â€“0.21 is reserved for the dynamic per-band pen row

    ax_prof.set_facecolor('#1a1a1a')
    ax_strip.set_facecolor('#1a1a1a')

    min_len_sl = Slider(ax_sl_ax, 'Min seg (mm)', 0.0, 0.5,
                        valinit=s['min_len_mm'], valstep=0.005, color='steelblue')

    btn = Button(ax_btn, 'Confirm', color='#1e6e1e', hovercolor='#2ea82e')
    btn.label.set_color('white')
    btn.label.set_fontsize(10)

    # â”€â”€ Per-band pen row: rebuilt whenever band count changes â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    def _rebuild_pen_row(nb, cl):
        saved = {}
        for b in range(len(bstate['tbs'])):
            level = _level_for_band(b)
            saved[b] = _clean_pen(
                bstate['tbs'][b].text.strip(),
                default=_default_pen_for_level(level),
            )
        for b, pen in sl.get('pen_map', {}).items():
            saved[int(b)] = _clean_pen(pen)
        for tb in bstate['tbs']:
            try:
                tb.disconnect_events()
            except Exception:
                pass
        for ax in bstate['patch_axes'] + bstate['tb_axes']:
            if ax in fig.axes:
                fig.delaxes(ax)
        bstate['patch_axes'].clear()
        bstate['tb_axes'].clear()
        bstate['tbs'].clear()

        bsw = 0.78 / nb
        tbw = min(max(bsw - 0.032, 0.05), 0.09)
        for b in range(nb):
            x0   = 0.07 + b * bsw
            level = _level_for_band(b)
            default_pen = saved.get(b, _default_pen_for_level(level))
            ax_p = fig.add_axes([x0, 0.06, 0.020, 0.15])
            ax_p.set_facecolor(_rgb_float(default_pen))
            ax_p.set_xticks([]); ax_p.set_yticks([])
            ax_p.text(0.5, 0.62, f"N{level}", transform=ax_p.transAxes,
                      ha='center', va='center', fontsize=8,
                      color='white', fontweight='bold')
            ax_p.text(0.5, 0.34, f"P{default_pen}", transform=ax_p.transAxes,
                      ha='center', va='center', fontsize=7,
                      color='white', fontweight='bold')
            for sp in ax_p.spines.values():
                sp.set_edgecolor('#666')
            bstate['patch_axes'].append(ax_p)

            ax_tb = fig.add_axes([x0 + 0.025, 0.08, tbw, 0.10])
            tb    = TextBox(ax_tb, f"Pen N{level}", initial=str(default_pen))
            def _update_pen_box(text, ax=ax_p, lvl=level):
                pen = _clean_pen(text, default=_default_pen_for_level(lvl))
                ax.set_facecolor(_rgb_float(pen))
                if len(ax.texts) > 1:
                    ax.texts[1].set_text(f"P{pen}")
                fig.canvas.draw_idle()
            tb.on_submit(_update_pen_box)
            bstate['tb_axes'].append(ax_tb)
            bstate['tbs'].append(tb)

    # â”€â”€ Full redraw: profile + strip; rebuild pen row only if band count changed
    def _draw():
        thr  = sorted(thresholds)
        nb   = len(thr) + 1
        cl   = [CMAP(b / max(nb - 1, 1)) for b in range(nb)]
        mln  = s['min_len_mm']
        bd   = _classify(avg, thr)
        segs = _build_segs_merged(bd, dist, mln)

        ax_prof.cla();  ax_prof.set_facecolor('#1a1a1a')
        ax_strip.cla(); ax_strip.set_facecolor('#1a1a1a')

        for b, si, ei in segs:
            ax_prof.plot(dist[si:ei], avg[si:ei],
                         color=cl[b] if b >= 0 else '#666', lw=1.5)
        for tv in thr:
            ax_prof.axhline(tv, color='white', lw=1.0, ls='--', alpha=0.75)
        ax_prof.set_xlabel("Distance along scan line (mm)", fontsize=9)
        ax_prof.set_ylabel(f"Height ({z_unit})", fontsize=9)
        ax_prof.set_title(
            f"Scan line {i + 1} of {len(scan_lines)}  |  "
            f"{nb} level band(s)  |  drag thresholds  |  type Pen 1-14 below",
            fontsize=9,
        )
        ax_prof.grid(True, alpha=0.2)

        for b, si, ei in segs:
            if b >= 0:
                ax_strip.fill_betweenx([0, 1], dist[si], dist[ei - 1],
                                       color=cl[b], alpha=0.9)
        ax_strip.set_xlim(0, dist[-1]); ax_strip.set_ylim(0, 1)
        ax_strip.set_yticks([])
        ax_strip.set_xlabel("Distance along scan line (mm)", fontsize=8)

        if nb != len(bstate['tbs']):
            _rebuild_pen_row(nb, cl)

        fig.canvas.draw_idle()

    _draw()

    # â”€â”€ Event handlers â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    def on_min_len(val):
        s['min_len_mm'] = val
        _draw()

    def on_press(event):
        if event.inaxes is not ax_prof or event.button != 1 or not thresholds:
            return
        y0, y1 = ax_prof.get_ylim()
        ds  = [abs(event.ydata - t) for t in thresholds]
        idx = int(np.argmin(ds))
        if ds[idx] <= 0.04 * abs(y1 - y0):
            drag['idx'] = idx

    def on_motion(event):
        if drag['idx'] is None or event.inaxes is not ax_prof:
            return
        thresholds[drag['idx']] = event.ydata
        _draw()

    def on_release(event):
        drag['idx'] = None

    def on_key(event):
        if event.key in ('+', '='):
            thr    = sorted(thresholds)
            bounds = [h_lo] + thr + [h_hi]
            gaps   = [(bounds[k + 1] - bounds[k], k) for k in range(len(bounds) - 1)]
            _, gi  = max(gaps)
            thresholds.append((bounds[gi] + bounds[gi + 1]) / 2)
            _draw()
        elif event.key in ('-', '_') and thresholds:
            thresholds.pop()
            _draw()

    def on_confirm(event):
        pen_map = {}
        for b_idx, tb in enumerate(bstate['tbs']):
            level = _level_for_band(b_idx)
            pen_map[b_idx] = _clean_pen(
                tb.text.strip(),
                default=_default_pen_for_level(level),
            )
        scan_lines[i]['pen_map']    = pen_map
        scan_lines[i]['level_pen_map'] = {
            _level_for_band(b): pen for b, pen in pen_map.items()
        }
        scan_lines[i]['thresholds'] = sorted(thresholds)
        scan_lines[i]['min_len_mm'] = s['min_len_mm']
        win['done'] = True
        plt.close(fig)

    min_len_sl.on_changed(on_min_len)
    fig.canvas.mpl_connect('button_press_event',   on_press)
    fig.canvas.mpl_connect('motion_notify_event',  on_motion)
    fig.canvas.mpl_connect('button_release_event', on_release)
    fig.canvas.mpl_connect('key_press_event',      on_key)
    btn.on_clicked(on_confirm)

    plt.show()
    while plt.fignum_exists(fig.number):
        plt.pause(0.05)

    if win['done']:
        pm = scan_lines[i]['pen_map']
        print(f"Scan line {i + 1}: " +
              "  ".join(
                  f"N{_level_for_band(b)} -> Pen {p} RGB {_pen_rgb(p)}"
                  for b, p in sorted(pm.items())
              ))
    else:
        print(f"Scan line {i + 1}: closed without confirming.")

# â”€â”€ Part 2: DXF export (single document, all scan lines) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
doc = ezdxf.new('R2010')
msp = doc.modelspace()
total = 0
skipped_short = 0
svg_segments = []

# Coordinate frame aligned to Line 1
line1_p1 = np.array(line1_coords['p1'], dtype=float)
line1_p2 = np.array(line1_coords['p2'], dtype=float)
u_x = (line1_p2 - line1_p1) / np.linalg.norm(line1_p2 - line1_p1)
u_y = np.array([-u_x[1], u_x[0]])  # 90Â° CCW from u_x
if np.dot(np.array(line2_coords['p2'], dtype=float) - line1_p1, u_y) < 0:
    u_y = -u_y  # flip so Line 2's lower endpoint is on the positive side

for i, sl in enumerate(scan_lines):
    if 'pen_map' not in sl:
        print(f"Scan line {i + 1}: skipped (pen map not confirmed).")
        continue

    pad_px    = sl.get('pad', 0)
    thr       = sl.get('thresholds', [])
    pen_map   = sl['pen_map']
    dist, avg = _sample_profile(sl, pad_px)
    if dist[-1] <= 0:
        print(f"Scan line {i + 1}: zero length, skipped.")
        continue

    band_arr = _classify(avg, thr)
    segs     = _build_segs_merged(band_arr, dist, sl.get('min_len_mm', 0.0))
    # Negate the perpendicular distance to flip image y-down to DXF y-up
    d_perp_i = -float(np.dot(np.array(sl['p1'], dtype=float) - line1_p1, u_y))

    for b, si, ei in segs:
        if b < 0:
            continue
        x0 = float(dist[si])
        x1 = float(dist[ei - 1])
        if (x1 - x0) < MIN_DXF_SEGMENT_MM:
            skipped_short += 1
            continue
        level = _level_for_band(b)
        pen = _clean_pen(pen_map.get(b), default=_default_pen_for_level(level))
        dxf_color = _pen_dxf_color(pen)
        layer_name = f"PEN_{pen}_NIVEL_{level}"
        if layer_name not in doc.layers:
            doc.layers.new(layer_name, dxfattribs={'color': dxf_color})
        msp.add_line(
            (x0, d_perp_i),
            (x1, d_perp_i),
            dxfattribs={'layer': layer_name, 'color': dxf_color},
        )
        svg_segments.append({
            'id': f"{layer_name}_LINE_{i + 1:02d}_SEG_{total + 1:04d}",
            'layer': layer_name,
            'pen': pen,
            'x0': x0,
            'y0': d_perp_i,
            'x1': x1,
            'y1': d_perp_i,
        })
        total += 1

out_path = experiment_dir / (region_name + '_profile.dxf')
svg_path = out_path.with_suffix('.svg')
doc.saveas(str(out_path))
write_svg_lines(svg_path, svg_segments)
print(f"\nDXF export: {total} segment(s) across {len(scan_lines)} scan line(s)")
print(f"Short segments skipped: {skipped_short}")
print(f"Saved DXF -> {out_path}")
print(f"Saved SVG -> {svg_path}")

Scan line 1: N2 -> Pen 5 RGB (255, 170, 0)  N3 -> Pen 6 RGB (0, 170, 170)  N4 -> Pen 7 RGB (85, 85, 85)
Scan line 2: N2 -> Pen 1 RGB (255, 0, 0)  N3 -> Pen 2 RGB (0, 255, 0)  N4 -> Pen 3 RGB (0, 0, 255)
Scan line 3: N2 -> Pen 1 RGB (255, 0, 0)  N3 -> Pen 2 RGB (0, 255, 0)  N4 -> Pen 3 RGB (0, 0, 255)
Scan line 4: N2 -> Pen 1 RGB (255, 0, 0)  N3 -> Pen 2 RGB (0, 255, 0)  N4 -> Pen 3 RGB (0, 0, 255)
Scan line 5: N2 -> Pen 9 RGB (0, 170, 0)  N3 -> Pen 10 RGB (255, 255, 0)  N4 -> Pen 11 RGB (255, 0, 255)

DXF export: 199 segment(s) across 5 scan line(s)
Short segments skipped: 0
Saved DXF -> C:\Users\javie\Desktop\Universidad\Verano 2026\laser_correction\Experiment 4 070126\Region 1\Region 1_profile.dxf
Saved SVG -> C:\Users\javie\Desktop\Universidad\Verano 2026\laser_correction\Experiment 4 070126\Region 1\Region 1_profile.svg
